In [ ]:
import os
import glob
import subprocess
import tempfile

from typing import Optional


# Use Homebrew ffmpeg/ffprobe explicitly.
FFMPEG_BIN = "/usr/local/bin/ffmpeg"
FFPROBE_BIN = "/usr/local/bin/ffprobe"

if not os.path.exists(FFMPEG_BIN):
    raise FileNotFoundError(f"ffmpeg not found at {FFMPEG_BIN}")
if not os.path.exists(FFPROBE_BIN):
    raise FileNotFoundError(f"ffprobe not found at {FFPROBE_BIN}")


In [2]:
%cd "/Users/henrik/Downloads/__oshi_no_ko"
os.getcwd()

/Users/henrik/Downloads/__oshi_no_ko


/usr/local/anaconda3/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


'/Users/henrik/Downloads/__oshi_no_ko'

In [3]:
def get_subtitle_codec(input_file: str, stream_index: int) -> str:
    """Robust codec detection for PGS + text subtitles"""
    try:
        if not FFPROBE_BIN:
            print("ffprobe not found (ffprobe/FFPROBE_BIN missing) - treating as unknown")
            return "unknown"

        # Primary attempt with high probe values
        cmd = [
            FFPROBE_BIN, '-v', 'quiet',
            '-probesize', '100M',
            '-analyzeduration', '200M',
            '-select_streams', f'0:s:{stream_index}',
            '-show_entries', 'stream=codec_name',
            '-of', 'default=noprint_wrappers=1:nokey=1',
            input_file
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        codec = result.stdout.strip()
        if codec:
            return codec

        # Fallback attempt
        cmd2 = [
            FFPROBE_BIN, '-v', 'quiet',
            '-probesize', '100M',
            '-analyzeduration', '200M',
            '-select_streams', f'0:s:{stream_index}',
            '-show_entries', 'stream=codec_name',
            '-of', 'csv=p=0',
            input_file
        ]
        result2 = subprocess.run(cmd2, capture_output=True, text=True, timeout=15)
        return result2.stdout.strip() or "unknown"
    except Exception as e:
        print(f"ffprobe detection failed: {e}")
        return "unknown"

In [ ]:
def burn_subtitles(
    input_file: str,
    output_file: str,
    subtitle_stream_index: Optional[int] = 0,
    audio_index: int = 0
) -> None:
    if subtitle_stream_index is None:
        print("(no subtitles requested)")
        return

    print(f"Using ffmpeg: {FFMPEG_BIN}")

    original_input = input_file
    clean_input = None

    try:
        # Temporary clean filename (avoids all escaping issues)
        base_dir = os.path.dirname(input_file) or "."
        ext = os.path.splitext(input_file)[1]
        with tempfile.NamedTemporaryFile(dir=base_dir, suffix=ext, delete=False) as tmp:
            clean_input = tmp.name

        os.rename(original_input, clean_input)
        print(f"Temporarily renamed → {os.path.basename(clean_input)}")

        # Detect subtitle type
        codec = get_subtitle_codec(clean_input, subtitle_stream_index)
        print(f"Subtitle codec detected: {codec}")

        codec_lower = codec.lower()
        is_pgs = ("pgs" in codec_lower) or (codec_lower == "hdmv_pgs_subtitle")

        base_command = [
            FFMPEG_BIN, '-y',
            '-probesize', '100M',
            '-analyzeduration', '200M',
            '-i', clean_input,
            '-map', '0:v:0',
            '-map', f'0:a:{audio_index}',
        ]

        encode_args = [
            '-c:v', 'mpeg4',
            '-q:v', '3',
            '-c:a', 'aac',
            '-b:a', '160k',
            '-movflags', '+faststart',
            output_file,
        ]

        def run_ffmpeg(extra_args: list, label: str) -> None:
            command = base_command + extra_args + encode_args
            print(label)
            subprocess.run(command, check=True, capture_output=True, text=True)

        def build_subtitles_filter(path: str, stream_index: int) -> str:
            # Use explicit option names for ffmpeg filter parser reliability.
            safe = path.replace("\\", "\\\\").replace(":", "\\:").replace("'", "\\'")
            return f"subtitles=filename='{safe}':stream_index={stream_index}"

        # Decide method: PGS/bitmap vs text subtitles.
        # If ffprobe can't identify the codec, prefer text first (much more common for your issue),
        # then fall back to the PGS overlay method.
        if is_pgs:
            print("(PGS bitmap subtitles detected → using overlay)")
            run_ffmpeg(
                [
                    '-filter_complex', f'[0:v:0][0:s:{subtitle_stream_index}]overlay[v]',
                    '-map', '[v]'
                ],
                f"Converting {os.path.basename(original_input)} → {os.path.basename(output_file)}",
            )
        elif codec != "unknown":
            print("(Text-based subtitles detected → using subtitles filter)")
            vf = build_subtitles_filter(clean_input, subtitle_stream_index)
            run_ffmpeg(
                ['-vf', vf],
                f"Converting {os.path.basename(original_input)} → {os.path.basename(output_file)}",
            )
        else:
            print("(Subtitle codec unknown → trying text subtitles first)")
            vf = build_subtitles_filter(clean_input, subtitle_stream_index)

            try:
                run_ffmpeg(
                    ['-vf', vf],
                    f"Converting {os.path.basename(original_input)} → {os.path.basename(output_file)}",
                )
            except subprocess.CalledProcessError as text_err:
                text_stderr = text_err.stderr.strip() if text_err.stderr else "(no stderr)"
                print("Text filter failed. First attempt stderr:")
                print(text_stderr)

                # Save first-attempt error details so fallback reasons are inspectable later.
                try:
                    log_file = os.path.join(base_dir, "subtitle_fallback_errors.log")
                    with open(log_file, "a", encoding="utf-8") as fh:
                        fh.write("\n" + "=" * 80 + "\n")
                        fh.write(f"input={original_input}\n")
                        fh.write(f"output={output_file}\n")
                        fh.write(f"subtitle_stream_index={subtitle_stream_index}\n")
                        fh.write(f"audio_index={audio_index}\n")
                        fh.write(f"detected_codec={codec}\n")
                        fh.write("first_attempt=text_subtitles_filter\n")
                        fh.write("stderr:\n")
                        fh.write(text_stderr + "\n")
                    print(f"Saved fallback details to {log_file}")
                except Exception as log_err:
                    print(f"Could not write fallback log: {log_err}")

                print("Text filter failed → retrying as PGS overlay")
                run_ffmpeg(
                    [
                        '-filter_complex', f'[0:v:0][0:s:{subtitle_stream_index}]overlay[v]',
                        '-map', '[v]'
                    ],
                    f"Converting {os.path.basename(original_input)} → {os.path.basename(output_file)}",
                )

        print("Done ✓")

    except subprocess.CalledProcessError as e:
        print("FFmpeg error:")
        print(e.stderr if e.stderr else str(e))
    except Exception as e:
        print(f"Error: {e}")
    finally:
        # Always restore original filename
        if clean_input and os.path.exists(clean_input):
            try:
                if os.path.exists(original_input):
                    os.unlink(original_input)
                os.rename(clean_input, original_input)
                print("Restored original filename")
            except Exception as restore_err:
                print(f"Restore warning: {restore_err}")

In [5]:
ext = "mkv"
audio_index = 0
subtitle_index = 0
files = sorted(glob.glob(f"*.{ext}"))
files

['Oshi no Ko S01E34 (3-11).mkv']

In [6]:
if not os.path.exists('_out/'):
    os.mkdir('_out')

In [ ]:
for file in files:

    file_out = "_out/" + file.replace(f".{ext}", '.mp4')

    # skip existing files
    if os.path.exists(file_out):
        print(f"skipping {file}")
        continue

    burn_subtitles(file, file_out, subtitle_index, audio_index)


Using ffmpeg: /usr/local/bin/ffmpeg
Temporarily renamed → tmp659h5hr2.mkv
Subtitle codec detected: unknown
(Subtitle codec unknown → trying text subtitles first)
Converting Oshi no Ko S01E34 (3-11).mkv → Oshi no Ko S01E34 (3-11).mp4


In [ ]:
# backup 

# burn with hard subs (text), default font size
# os.system(f"ffmpeg -i \"{file}\" -map 0:v:0 -map 0:a:0 -vf \"scale=-1:720,  subtitles='{file}':stream_index=0'\" \"{file_out}\"")
# os.system(f"ffmpeg -i \"{file}\" -map 0:v:0 -map 0:a:0 -vf \"scale=-1:720, subtitles='{file}':si=2'\" \"{file_out}\"")

# ass
# os.system(f"ffmpeg -i \"{file}\" -map 0:v:0 -map 0:a:0 -vf \"scale=-1:720, ass='{file}'\" \"{file_out}\"")

# PGS
# os.system(f"ffmpeg -i \"{file}\" -filter_complex \"[0:v:0][0:s:0]overlay=x=0:y=0[v]\" -map \"[v]\" -map 0:a:1 \"{file_out}\"")    
# os.system(f"ffmpeg -i \"{file}\" -filter_complex \"[0:s:0]scale=iw/2:-1[vsub];[0:v][vsub]overlay=x=0:y=180[v]\" -map \"[v]\" -map 0:a:1 \"{file_out}\"")


In [ ]:
1

1